# DataSaathi on Google Colab (with Gemma 4)

Ask campus electricity and water data questions in English, Hindi or Hinglish. Gemma 4 plans, code calculates, every answer has a Proof Receipt.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then run the cells below in order (about 10 to 15 minutes the first time, mostly the model download).

| Model (set in cell 1) | Download | Fits free T4 (15 GB)? | Notes |
|---|---|---|---|
| `gemma4:e4b` (default) | ~9.6 GB | Yes | The model we validated: 20/20 on our test set |
| `gemma4:12b` | ~7.6 GB | Yes | Larger dense model; try it and compare with the eval cell |
| `gemma4:26b` | ~18 GB | No (needs a 24 GB GPU such as L4 or A100) | MoE, higher quality |

In [ ]:
# 1) Settings
MODEL = "gemma4:e4b"        # or "gemma4:12b" / "gemma4:26b" (26b needs a 24 GB GPU)
REPO_URL = "https://github.com/dhruvsoran/datasaathi.git"
STT_MODEL = "small"          # speech model for voice questions: "small" (fast) or "medium" (better for Indian languages)


In [ ]:
# 2) Check the GPU
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print(out or "NO GPU FOUND: Runtime > Change runtime type > T4 GPU, then reconnect.")

In [ ]:
# 3) Install Ollama
!apt-get -y -qq install zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version


In [ ]:
# 4) Start Ollama and download the model
import subprocess, time, urllib.request
subprocess.Popen(["ollama", "serve"], stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(40):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434", timeout=2); break
    except Exception:
        time.sleep(1)
!ollama pull {MODEL}
!ollama list
!ollama show {MODEL}      # prints architecture, parameters, context length and quantisation


In [ ]:
# 5) Get the project code (clones from GitHub; falls back to uploading the zip)
import os
if not os.path.exists("/content/datasaathi/app.py"):
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/datasaathi"], capture_output=True, text=True)
    if r.returncode != 0:
        print("Clone failed, upload DataSaathi_Project.zip instead:")
        from google.colab import files
        name = list(files.upload().keys())[0]
        !unzip -oq "/content/{name}" -d /content
os.chdir("/content/datasaathi")
!pip install -q -r requirements.txt
print("Ready:", os.path.exists("app.py"))
!pip install -q -r requirements-voice.txt     # voice questions (Whisper speech-to-text)


In [ ]:
# 6) Prove Gemma 4 works: one plan in English, one in Hindi, then the 20-question evaluation
import os
os.environ["DATASAATHI_MODEL"] = MODEL
!python scripts/check_gemma.py
!python evals/run_evals.py --gemma

In [ ]:
# 7) Launch the app and get a public link, then open it
import os
os.environ["DATASAATHI_MODEL"] = MODEL
os.environ["DATASAATHI_STT_MODEL"] = STT_MODEL
!python scripts/colab_launch.py

**In the app:** the sidebar should say *Gemma 4 connected* and the header *Gemma 4 · gemma4:e4b*. Try the questions in `PITCH.md`, and upload `data/` sample files or your own CSV.

Free Colab sessions disconnect when idle. Keep this tab open while presenting.

**Voice:** open *Ask by voice*, allow the microphone, speak, press stop, check the text it heard, then press *Ask this question*. The first voice question downloads the speech model (about 0.5 to 1.5 GB).